# Panda <-> GitHub: chạy code mới nhất, đẩy kết quả lên GitHub

Notebook này chỉ có 4 ô. Code nằm trên GitHub (`anphammmm/piezo-topology-synthesis`). Ô 1 lấy bản mới
nhất về, ô 2 chạy việc trên Panda (job LTspice, không dùng GPU), ô 3 xem tiến độ, ô 4 đẩy kết quả gọn
(báo cáo, CSV, log, không có waveform) lên nhánh `results`. Claude trên laptop đọc kết quả từ
nhánh đó, sửa code, push. Bạn chạy lại ô 1 là có bản sửa.

**Đặt notebook này ở ngoài thư mục repo**, ví dụ `/work/hpham/panda_sync.ipynb`. Jupyter lưu output vào
file notebook; nếu nó nằm trong repo thì `git pull` sẽ bị chặn vì file đã bị sửa.

Lần đầu: tạo file token theo `docs/panda-sync.md` (bước 1-2), rồi chạy ô 1.


## 1. Lấy code mới nhất

In [ ]:
import os, sys, json, glob, time, subprocess, importlib
from pathlib import Path

ME = "ltspice/claude"                        # <-- thư mục của bạn trong /work
GITHUB_USER = "anphammmm"
URL = "https://github.com/anphammmm/piezo-topology-synthesis.git"
BRANCH = "feat/v2-screen-size-paper"
REPO = Path(f"/work/{ME}/piezo-topology-synthesis")
SECRETS = Path(f"/work/{ME}/.secrets")
TOKEN_FILE = SECRETS / "github_token"

ENV = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if TOKEN_FILE.exists():                         # needed only to publish results (cell 4); the repo is public
    ASKPASS = SECRETS / "askpass.sh"            # git gets the token from here; it is never printed
    ASKPASS.write_text('#!/bin/sh\ncase "$1" in Username*) echo "$GITHUB_USER" ;; *) cat "$PIEZO_TOKEN_FILE" ;; esac\n')
    ASKPASS.chmod(0o700)
    ENV.update(GIT_ASKPASS=str(ASKPASS), GITHUB_USER=GITHUB_USER, PIEZO_TOKEN_FILE=str(TOKEN_FILE))
else:
    print(f"(chưa có token ở {TOKEN_FILE}: lấy code và chạy được, nhưng ô 4 sẽ không đẩy kết quả lên được)")

def sh(*cmd, cwd=REPO, check=True, timeout=300):
    print("$", " ".join(cmd[:3]), "...", flush=True)
    try:
        p = subprocess.run(cmd, cwd=cwd, env=ENV, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"'{' '.join(cmd[:3])}' took more than {timeout} s and was stopped "
                           "(network to GitHub, or a git lock left by an interrupted run: see docs/panda-sync.md)")
    out = (p.stdout + p.stderr).strip()
    if out:
        print(out[-3000:])
    if check and p.returncode != 0:
        raise RuntimeError(f"failed: {' '.join(cmd)}")
    return p

if not REPO.exists():
    sh("git", "clone", "--quiet", URL, str(REPO), cwd=REPO.parent)
sh("git", "config", "user.name", f"{GITHUB_USER} (Panda)")
sh("git", "config", "user.email", f"{GITHUB_USER}@users.noreply.github.com")
sh("git", "fetch", "--quiet", "origin")
sh("git", "checkout", "--quiet", BRANCH)
sh("git", "pull", "--ff-only", "--quiet", "origin", BRANCH)
sh("git", "log", "-1", "--format=code: %h %s (%cr)")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))               # the repository has no gpu.py, so nothing is shadowed
# drop every cached module of the repository, so the code just pulled is what runs (reloading a few
# modules kept stale ones such as piezo_synth.spec)
for m in [m for m in sys.modules if m in ("piezo_synth", "tools") or m.startswith(("piezo_synth.", "tools."))]:
    del sys.modules[m]
import piezo_synth.experiments, tools.panda_submit, tools.panda_campaign
from gpu import jobs, log, cancel

def submit(name, cmd, timeout=4 * 3600):
    tools.panda_submit.main(["--cwd", str(REPO), "--name", name, "--timeout", str(timeout), "--", cmd])
print("ready")

## 2. Chạy việc

Đặt `TASK` rồi chạy ô. Mỗi lần chỉ nên chạy một việc có LTspice (chúng dùng chung Wine).
- `"smoke"`: kiểm tra nhanh, khoảng 2 phút (chạy đầu tiên, và sau mỗi lần đổi mô hình linh kiện).
- `"single"`: một lần tìm kiếm đầy đủ, khoảng 30-60 phút.
- `"waves"`: waveform của thiết kế tốt nhất trong lần `single` gần nhất.
- `"campaign"`: toàn bộ thí nghiệm cho paper (8 biến thể x 5 seed, vài tiếng; chạy lại thì chạy tiếp).
- `"random_rerun"`: chạy lại baseline random với đủ ngân sách 3000 mô phỏng. Các run random cũ được chuyển
  (không xóa) sang `runs/exp_superseded/`.
- `"speccheck"`: kiểm tra 5 spec (P1-P5) không cần Gemma: FBR, mạch sách giáo khoa (có sizing) và witness
  trên từng spec, khoảng 1 tiếng. Chạy cái này TRƯỚC `"bench"`.
- `"bench"`: benchmark cho paper: 5 spec x 5 biến thể x 5 seed, mỗi spec một job (khoảng 1 ngày; chạy lại
  thì chạy tiếp).
- `"neutral"`: kiểm tra prompt có mớm đáp án không: biến thể `neutral_prompt` (prompt không có nhánh SSHI
  trong ví dụ JSON, không có công thức ton, không có ví dụ cầu chỉnh lưu/zener kẹp áp) trên P1 và P3,
  5 seed mỗi spec (khoảng nửa ngày). Chạy SAU khi `"bench"` xong.

In [ ]:
TASK = "smoke"
stamp = time.strftime("%Y%m%d-%H%M%S")
if TASK == "smoke":
    submit("piezo-smoke", "rm -rf runs/smoke && python3 -u -m piezo_synth.run --smoke --out runs/smoke", 3600)
elif TASK == "single":
    submit("piezo-single", f"python3 -u -m piezo_synth.run --out runs/single-{stamp}")
elif TASK == "waves":
    sel = sorted(REPO.glob("runs/single-*/selected/circuit.json"))
    assert sel, "chưa có lần chạy single nào hoàn tất"
    run_dir = sel[-1].parent.parent.relative_to(REPO).as_posix()
    submit("piezo-waves",
           f"python3 -u -m piezo_synth.reproduce {run_dir}/selected/circuit.json --out {run_dir}/waves-selected "
           f"--conditions S1,S4 --waveforms && python3 -u -m piezo_synth.reproduce "
           f"{run_dir}/candidates/c0000/circuit.json --out {run_dir}/waves-reference --conditions S1,S4 --waveforms",
           1800)
elif TASK == "campaign":
    tools.panda_campaign.main(["--cwd", str(REPO)])
elif TASK == "speccheck":
    submit("piezo-speccheck", "python3 -u -m piezo_synth.speccheck --out runs/speccheck", 4 * 3600)
elif TASK == "bench":
    tools.panda_campaign.main(["--cwd", str(REPO), "--bench"])
elif TASK == "neutral":
    tools.panda_campaign.main(["--cwd", str(REPO), "--bench", "--variants", "neutral_prompt", "--specs", "P1,P3"])
elif TASK == "random_rerun":
    import shutil
    old = REPO / "runs/exp/random"
    if old.exists():
        dest = REPO / f"runs/exp_superseded/random-{stamp}"
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(old), str(dest))
        for f in REPO.glob("runs/exp/random-seed*.log"):
            shutil.move(str(f), str(dest / f.name))
        print(f"old random runs moved to {dest.relative_to(REPO)}")
    tools.panda_campaign.main(["--cwd", str(REPO), "--variants", "random"])
else:
    raise ValueError(TASK)

## 3. Tiến độ (chạy bất cứ lúc nào)

In [ ]:
jobs()
for f in sorted(REPO.glob("runs/logs/*.log"), key=lambda p: p.stat().st_mtime)[-2:]:
    print(f"\n===== {f.relative_to(REPO)} (last lines) =====")
    print("\n".join(f.read_text(errors="replace").splitlines()[-15:]))
done = len(glob.glob(str(REPO / "runs/exp/*/seed*/outcome.json")))
if (REPO / "runs/exp").exists():
    print(f"\ncampaign: {done} of 40 runs finished")
bench = len(glob.glob(str(REPO / "runs/bench/*/*/seed*/outcome.json")))
if (REPO / "runs/bench").exists():
    print(f"benchmark: {bench} runs finished (125 for the 5 specs x 5 variants, +10 with neutral)")
if (REPO / "runs/speccheck/speccheck.md").exists():
    print((REPO / "runs/speccheck/speccheck.md").read_text())

## 4. Đẩy kết quả lên GitHub

Chạy khi một việc đã xong (hoặc giữa chừng: kết quả dở dang cũng đẩy được). Nếu đã có campaign, ô này
tổng hợp các bảng cho paper trước. Viết vài chữ vào `MESSAGE` để Claude biết đây là kết quả gì.

In [ ]:
MESSAGE = "smoke test"
if (REPO / "runs/exp").exists():
    sh(sys.executable, "-m", "piezo_synth.experiments", "runs/exp", "--paper", "runs/paper_generated",
       "--smoke", "runs/smoke", check=False)
    sh(sys.executable, "tools/paper_figures.py", "runs/exp", "runs/paper_generated", check=False)
if (REPO / "runs/bench").exists():
    sh(sys.executable, "-m", "piezo_synth.experiments", "runs/bench", "--bench", "--paper",
       "runs/paper_generated", check=False)
if TOKEN_FILE.exists():
    sh(sys.executable, "tools/publish_results.py", "--message", MESSAGE, check=False, timeout=1800)
else:
    print(f"Chưa có token ({TOKEN_FILE}), nên không đẩy lên GitHub được.")
    print("Cách khác: chạy ô 3 rồi dán output cho Claude. Hoặc tạo token theo docs/panda-sync.md (bước 1-2).")